# Preprocessing CBIS-DDSM dataset

# Important libraries

# Preprocessing class with all needed functions

-------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------

In [1]:
pip install pydicom

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import pydicom
import numpy as np
import pandas as pd
import cv2

# Calc Test Loading & Transforming to PNG format (simple processing)

In [4]:
# B M 2
import os
import pandas as pd
import numpy as np
import pydicom
import cv2

# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_test_set_png_pathology"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv(
    "D:/cbis-ddsm_dataset_licenta/data/raw/calc_case_description_test_set.csv"
)
metadata_df = pd.read_csv(
    "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv"
)

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(
    columns={
        "image file path": "full_path",
        "cropped image file path": "roi_path",
        "ROI mask file path": "mask_path",
    }
)

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Main Processing Loop ===
for idx, row in test_set_df.iterrows():
    patient_id = row["patient_id"]
    pathology = str(row["pathology"]).strip().lower()

    # Normalize pathology naming
    if "malig" in pathology:
        pathology_folder = "malignant"
    elif "benign w" in pathology:
        pathology_folder = "benign_w_callback"
    elif "benign" in pathology:
        pathology_folder = "benign"
    else:
        pathology_folder = "unknown"

    # Append pathology to patient folder name
    patient_label = f"{patient_id}_{pathology_folder}"

    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue

        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)

        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        folder_type = get_image_type(file_location)

        # Create per-patient output dirs
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_label)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_label)
        else:
            patient_save_dir = os.path.join(MASK_DIR, patient_label)

        os.makedirs(patient_save_dir, exist_ok=True)

        # Process DICOMs
        if folder_type == "full":
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dicom_to_png(os.path.join(full_file_dir, dcm_file), save_path)

        elif folder_type == "roi_mask":
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)

                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

print("[INFO] Processing completed successfully!")


[INFO] Processing completed successfully!


In [1]:
## B VS M added
import os
import pandas as pd
import numpy as np
import pydicom
import cv2

# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_test_set_png_b_vs_m"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/calc_case_description_test_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row["patient_id"]
    b_or_m_label = row["pathology"].strip().replace(" ", "_")  # clean label for folder name

    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue

        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)

        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        folder_type = get_image_type(file_location)

        # === CREATE LABELLED SUBFOLDER ===
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, f"{patient_id}_{b_or_m_label}")
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, f"{patient_id}_{b_or_m_label}")
        else:
            patient_save_dir = os.path.join(MASK_DIR, f"{patient_id}_{b_or_m_label}")

        os.makedirs(patient_save_dir, exist_ok=True)

        # === PROCESS FILES ===
        if folder_type == "full":
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dicom_to_png(os.path.join(full_file_dir, dcm_file), save_path)

        elif folder_type == "roi_mask":
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)

                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

print("[INFO] Processing completed!")


[INFO] Processing completed!


In [3]:
# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_test_set_png"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/calc_case_description_test_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    """
    Determine if the folder contains full mammogram, ROI, or mask images
    based on the folder description in the path
    """
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"  # This folder contains both ROI and mask
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row['patient_id']
    
    # Process each path type
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue
            
        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)
        
        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()  # Ensure consistent ordering
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        # Determine the image type from folder description
        folder_type = get_image_type(file_location)
        
        # Create patient subfolder
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_id)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_id)
        else:  # mask
            patient_save_dir = os.path.join(MASK_DIR, patient_id)
            
        os.makedirs(patient_save_dir, exist_ok=True)

        # Process files based on type
        if folder_type == "full":
            # Full mammogram - process all files
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dcm_path = os.path.join(full_file_dir, dcm_file)
                
                dicom_to_png(dcm_path, save_path)
                    
        elif folder_type == "roi_mask":
            # ROI/Mask folder - separate based on file naming
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                
                # Check if this is ROI (1-1) or mask (1-2) based on filename
                if dcm_file.endswith("1-1.dcm"):
                    # This is ROI
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                        os.makedirs(os.path.join(ROI_CROP_DIR, patient_id), exist_ok=True)
                        
                        dicom_to_png(dcm_path, save_path)
                            
                elif dcm_file.endswith("1-2.dcm"):
                    # This is mask
                    if path_type == "mask":
                        output_name = f"{subject_id}_mask.png"
                        save_path = os.path.join(MASK_DIR, patient_id, output_name)
                        os.makedirs(os.path.join(MASK_DIR, patient_id), exist_ok=True)
                        
                        dicom_to_png(dcm_path, save_path)

print("[INFO] Processing completed!")

[INFO] Processing completed!


In [1]:
# MASK RESIZING
import os
import cv2
import pydicom
import numpy as np
import pandas as pd

# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_test_set_png_masks_resized"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")
PROCESSED_MASK_DIR = os.path.join(OUTPUT_ROOT, "processed_cropped_masks")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)
os.makedirs(PROCESSED_MASK_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/calc_case_description_test_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row['patient_id']
    
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue
            
        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)
        
        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        folder_type = get_image_type(file_location)

        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_id)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_id)
        else:
            patient_save_dir = os.path.join(MASK_DIR, patient_id)
            
        os.makedirs(patient_save_dir, exist_ok=True)

        # === Convert DICOMs ===
        if folder_type == "full":
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dicom_to_png(os.path.join(full_file_dir, dcm_file), save_path)
                    
        elif folder_type == "roi_mask":
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                    os.makedirs(os.path.join(ROI_CROP_DIR, patient_id), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.join(MASK_DIR, patient_id), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)

print("[INFO] DICOM conversion completed!")

# === Step 2: Crop + resize masks ===
print("[INFO] Starting mask cropping and resizing...")

for patient in os.listdir(MASK_DIR):
    patient_mask_dir = os.path.join(MASK_DIR, patient)
    patient_roi_dir = os.path.join(ROI_CROP_DIR, patient)
    if not os.path.isdir(patient_mask_dir):
        continue

    patient_output_dir = os.path.join(PROCESSED_MASK_DIR, patient)
    os.makedirs(patient_output_dir, exist_ok=True)

    for mask_filename in os.listdir(patient_mask_dir):
        if not mask_filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        mask_path = os.path.join(patient_mask_dir, mask_filename)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            print(f"[WARN] Could not load mask: {mask_path}")
            continue

        coords = cv2.findNonZero(mask)
        if coords is None:
            print(f"[WARN] Empty mask in {mask_filename}")
            continue

        x, y, w, h = cv2.boundingRect(coords)
        cropped_mask = mask[y:y+h, x:x+w]

        # Find matching ROI file
        roi_filename = mask_filename.replace("_mask", "_roi")
        roi_path = os.path.join(patient_roi_dir, roi_filename)

        if not os.path.exists(roi_path):
            print(f"[WARN] ROI not found for mask: {mask_filename}")
            continue

        roi_img = cv2.imread(roi_path, cv2.IMREAD_GRAYSCALE)
        if roi_img is None:
            print(f"[WARN] Could not load ROI: {roi_path}")
            continue

        resized_mask = cv2.resize(cropped_mask, (roi_img.shape[1], roi_img.shape[0]), interpolation=cv2.INTER_NEAREST)

        save_path = os.path.join(patient_output_dir, mask_filename)
        cv2.imwrite(save_path, resized_mask)

        print(f"[OK] Saved cropped+resized mask: {save_path}")

print("[INFO] All masks cropped and resized successfully!")


[INFO] DICOM conversion completed!
[INFO] Starting mask cropping and resizing...
[OK] Saved cropped+resized mask: D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_test_set_png_masks_resized\processed_cropped_masks\P_00038\Calc-Test_P_00038_LEFT_CC_1_mask.png
[OK] Saved cropped+resized mask: D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_test_set_png_masks_resized\processed_cropped_masks\P_00038\Calc-Test_P_00038_LEFT_MLO_1_mask.png
[OK] Saved cropped+resized mask: D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_test_set_png_masks_resized\processed_cropped_masks\P_00038\Calc-Test_P_00038_RIGHT_CC_1_mask.png
[OK] Saved cropped+resized mask: D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_test_set_png_masks_resized\processed_cropped_masks\P_00038\Calc-Test_P_00038_RIGHT_CC_2_mask.png
[OK] Saved cropped+resized mask: D:/cbis

# Calc Training Loading & Transforming to PNG format (complex processing)

In [6]:
# === CALC TRAIN SET (Pathology-labelled, B M 2 style) ===
import os
import pandas as pd
import numpy as np
import pydicom
import cv2

# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_train_set_png_pathology"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
train_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/calc_case_description_train_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize metadata paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)

# Rename key columns for easier access
train_set_df = train_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type ===
def get_image_type(file_location):
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === MAIN LOOP ===
for idx, row in train_set_df.iterrows():
    patient_id = row["patient_id"]
    pathology = str(row["pathology"]).strip().lower()

    # Normalize pathology folder names
    if "malig" in pathology:
        pathology_folder = "malignant"
    elif "benign w" in pathology:
        pathology_folder = "benign_w_callback"
    elif "benign" in pathology:
        pathology_folder = "benign"
    else:
        pathology_folder = "unknown"

    # Final patient folder name
    patient_label = f"{patient_id}_{pathology_folder}"

    # Process each of the three DICOM types
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue

        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)

        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        folder_type = get_image_type(file_location)

        # Define save directory per pathology-aware patient
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_label)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_label)
        else:
            patient_save_dir = os.path.join(MASK_DIR, patient_label)

        os.makedirs(patient_save_dir, exist_ok=True)

        # === Process DICOM files ===
        if folder_type == "full":
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dicom_to_png(os.path.join(full_file_dir, dcm_file), save_path)

        elif folder_type in ["roi_mask", "roi", "mask"]:
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                folder_lower = file_location.lower()

                # --- ROI (cropped images) ---
                if any(kw in folder_lower for kw in ["cropped images", "cropped_image", "cropped"]):
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(patient_save_dir, output_name)
                        dicom_to_png(dcm_path, save_path)
                        continue

                # --- MASK (ROI mask images) ---
                if "roi mask" in folder_lower and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)
                    continue

                # --- Fallback for 1-1 / 1-2 naming ---
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

print("[INFO] Processing completed successfully!")


[INFO] Processing completed successfully!


In [4]:
# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_train_set_png"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/calc_case_description_train_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    """
    Determine if the folder contains full mammogram, ROI, or mask images
    based on the folder description in the path
    """
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"  # This folder contains both ROI and mask
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row['patient_id']
    
    # Process each path type
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue
            
        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)
        
        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()  # Ensure consistent ordering
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        # Determine the image type from folder description
        folder_type = get_image_type(file_location)
        
        # Create patient subfolder
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_id)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_id)
        else:  # mask
            patient_save_dir = os.path.join(MASK_DIR, patient_id)
            
        os.makedirs(patient_save_dir, exist_ok=True)

        # Process files based on type
        if folder_type == "full":
            # Full mammogram - process all files
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dcm_path = os.path.join(full_file_dir, dcm_file)

                dicom_to_png(dcm_path, save_path)
                    
        elif folder_type in ["roi_mask", "roi", "mask"]:
            # Handle ROI, mask, or mixed folder cases
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                folder_lower = file_location.lower()
        
                # --- ROI (cropped images) ---
                if any(kw in folder_lower for kw in ["cropped images", "cropped_image", "cropped"]):
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                        os.makedirs(os.path.dirname(save_path), exist_ok=True)
                        dicom_to_png(dcm_path, save_path)
                        continue
        
                # --- MASK (ROI mask images) ---
                if path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
                    continue
        
                # --- Fallback (old naming 1-1 / 1-2) ---
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
        
                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)



print("[INFO] Processing completed!")


[INFO] Processing completed!


# Mass Test Loading & Transforming to PNG format (complex processing)

In [5]:
# === MASS TEST SET (Pathology-labelled, B M 2 style) ===
import os
import pandas as pd
import numpy as np
import pydicom
import cv2

# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/mass_case_description_test_set_png_pathology"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/mass_case_description_test_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === MAIN PROCESSING LOOP ===
for idx, row in test_set_df.iterrows():
    patient_id = row["patient_id"]
    pathology = str(row["pathology"]).strip().lower()

    # Normalize pathology naming
    if "malig" in pathology:
        pathology_folder = "malignant"
    elif "benign w" in pathology:
        pathology_folder = "benign_w_callback"
    elif "benign" in pathology:
        pathology_folder = "benign"
    else:
        pathology_folder = "unknown"

    # Append pathology to patient folder name
    patient_label = f"{patient_id}_{pathology_folder}"

    # Process all DICOM paths per case
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue

        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)

        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        folder_type = get_image_type(file_location)

        # Define save directories per pathology-aware patient
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_label)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_label)
        else:
            patient_save_dir = os.path.join(MASK_DIR, patient_label)

        os.makedirs(patient_save_dir, exist_ok=True)

        # === Process DICOMs ===
        if folder_type == "full":
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dicom_to_png(os.path.join(full_file_dir, dcm_file), save_path)

        elif folder_type in ["roi_mask", "roi", "mask"]:
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                folder_lower = file_location.lower()

                # --- ROI (cropped images) ---
                if any(kw in folder_lower for kw in ["cropped images", "cropped_image", "cropped"]):
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(patient_save_dir, output_name)
                        dicom_to_png(dcm_path, save_path)
                        continue

                # --- MASK (ROI mask images) ---
                if "roi mask" in folder_lower and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)
                    continue

                # --- Fallback for 1-1 / 1-2 naming ---
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

print("[INFO] Processing completed successfully!")


[INFO] Processing completed successfully!


In [2]:
# B vs M
import os
import pandas as pd
import numpy as np
import pydicom
import cv2

# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/mass_case_description_test_set_png_b_vs_m"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/mass_case_description_test_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row["patient_id"]
    b_or_m_label = row["pathology"].strip().replace(" ", "_")  # e.g., "benign_with_calcification"

    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue

        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)

        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        folder_type = get_image_type(file_location)

        # === CREATE LABELLED SUBFOLDER ===
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, f"{patient_id}_{b_or_m_label}")
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, f"{patient_id}_{b_or_m_label}")
        else:
            patient_save_dir = os.path.join(MASK_DIR, f"{patient_id}_{b_or_m_label}")

        os.makedirs(patient_save_dir, exist_ok=True)

        # === PROCESS FILES ===
        if folder_type == "full":
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{idx}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dicom_to_png(os.path.join(full_file_dir, dcm_file), save_path)

        elif folder_type in ["roi_mask", "roi", "mask"]:
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                folder_lower = file_location.lower()

                # --- ROI (cropped images) ---
                if any(kw in folder_lower for kw in ["cropped images", "cropped_image", "cropped"]):
                    if path_type == "roi":
                        output_name = f"{subject_id}_{idx}_roi.png"
                        save_path = os.path.join(patient_save_dir, output_name)
                        dicom_to_png(dcm_path, save_path)
                        continue

                # --- MASK (ROI mask images) ---
                if "roi mask images" in folder_lower and path_type == "mask":
                    output_name = f"{subject_id}_{idx}_mask.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)
                    continue

                # --- Fallback for older 1-1 / 1-2 naming ---
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_{idx}_roi.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_{idx}_mask.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

print("[INFO] Processing completed!")


[INFO] Processing completed!


In [4]:
# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/mass_case_description_test_set_png"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/mass_case_description_test_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    """
    Determine if the folder contains full mammogram, ROI, or mask images
    based on the folder description in the path
    """
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"  # This folder contains both ROI and mask
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row['patient_id']
    
    # Process each path type
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue
            
        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)
        
        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()  # Ensure consistent ordering
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        # Determine the image type from folder description
        folder_type = get_image_type(file_location)
        
        # Create patient subfolder
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_id)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_id)
        else:  # mask
            patient_save_dir = os.path.join(MASK_DIR, patient_id)
            
        os.makedirs(patient_save_dir, exist_ok=True)

        # Process files based on type
        if folder_type == "full":
            # Full mammogram - process all files
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dcm_path = os.path.join(full_file_dir, dcm_file)

                dicom_to_png(dcm_path, save_path)
                    
        elif folder_type in ["roi_mask", "roi", "mask"]:
            # Handle ROI, mask, or mixed folder cases
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                folder_lower = file_location.lower()
        
                # --- ROI (cropped images) ---
                if any(kw in folder_lower for kw in ["cropped images", "cropped_image", "cropped"]):
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                        os.makedirs(os.path.dirname(save_path), exist_ok=True)
                        dicom_to_png(dcm_path, save_path)
                        continue
        
                # --- MASK (ROI mask images) ---
                if path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
                    continue
        
                # --- Fallback (old naming 1-1 / 1-2) ---
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
        
                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)



print("[INFO] Processing completed!")


[INFO] Processing completed!


# Mass Training Loading & Transforming to PNG format (complex processing)

In [7]:
# === MASS TRAIN SET (Pathology-labelled, B M 2 style) ===
import os
import pandas as pd
import numpy as np
import pydicom
import cv2

# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/mass_case_description_train_set_png_pathology"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
train_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/mass_case_description_train_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize metadata paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)

# Rename columns for easier access
train_set_df = train_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === MAIN LOOP ===
for idx, row in train_set_df.iterrows():
    patient_id = row["patient_id"]
    pathology = str(row["pathology"]).strip().lower()

    # Normalize pathology naming
    if "malig" in pathology:
        pathology_folder = "malignant"
    elif "benign w" in pathology:
        pathology_folder = "benign_w_callback"
    elif "benign" in pathology:
        pathology_folder = "benign"
    else:
        pathology_folder = "unknown"

    # Combine patient ID + pathology for folder naming
    patient_label = f"{patient_id}_{pathology_folder}"

    # Process each path type (full, roi, mask)
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue

        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)

        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        folder_type = get_image_type(file_location)

        # Define save directories per pathology
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_label)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_label)
        else:
            patient_save_dir = os.path.join(MASK_DIR, patient_label)

        os.makedirs(patient_save_dir, exist_ok=True)

        # === Process DICOM files ===
        if folder_type == "full":
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dicom_to_png(os.path.join(full_file_dir, dcm_file), save_path)

        elif folder_type in ["roi_mask", "roi", "mask"]:
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                folder_lower = file_location.lower()

                # --- ROI (cropped images) ---
                if any(kw in folder_lower for kw in ["cropped images", "cropped_image", "cropped"]):
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(patient_save_dir, output_name)
                        dicom_to_png(dcm_path, save_path)
                        continue

                # --- MASK (ROI mask images) ---
                if "roi mask" in folder_lower and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)
                    continue

                # --- Fallback for older DICOM naming (1-1, 1-2) ---
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(patient_save_dir, output_name)
                    dicom_to_png(dcm_path, save_path)

print("[INFO] Processing completed successfully!")


[INFO] Processing completed successfully!


In [ ]:
# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/mass_case_description_train_set_png"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/mass_case_description_train_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    """
    Determine if the folder contains full mammogram, ROI, or mask images
    based on the folder description in the path
    """
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"  # This folder contains both ROI and mask
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row['patient_id']
    
    # Process each path type
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue
            
        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)
        
        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()  # Ensure consistent ordering
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        # Determine the image type from folder description
        folder_type = get_image_type(file_location)
        
        # Create patient subfolder
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_id)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_id)
        else:  # mask
            patient_save_dir = os.path.join(MASK_DIR, patient_id)
            
        os.makedirs(patient_save_dir, exist_ok=True)

        # Process files based on type
        if folder_type == "full":
            # Full mammogram - process all files
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dcm_path = os.path.join(full_file_dir, dcm_file)

                dicom_to_png(dcm_path, save_path)
                    
        elif folder_type in ["roi_mask", "roi", "mask"]:
            # Handle ROI, mask, or mixed folder cases
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                folder_lower = file_location.lower()
        
                # --- ROI (cropped images) ---
                if any(kw in folder_lower for kw in ["cropped images", "cropped_image", "cropped"]):
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                        os.makedirs(os.path.dirname(save_path), exist_ok=True)
                        dicom_to_png(dcm_path, save_path)
                        continue
        
                # --- MASK (ROI mask images) ---
                if path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
                    continue
        
                # --- Fallback (old naming 1-1 / 1-2) ---
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
        
                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)



print("[INFO] Processing completed!")
